In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="KrIa1A2HfnVXAbCFYT2E")
project = rf.workspace("s-workspace-ip0im").project("stanford-dogs-dataset-dog-breed-6entu")
version = project.version(1)
dataset = version.download("yolov8")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 85.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.15
    Uninstalling idna-3.15:
      Successfully uninstalled idna-3.15
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Stanford-Dogs-Dataset-dog-breed-1 in yolov8:: 100%|██████████| 40987/40987 [00:10<00:00, 3732.12it/s]


In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 51.9 MB/s eta 0:00:00


In [ ]:
!unzip -q /content/cat_images.zip -d /content/mydataset/cat

[/content/cat_images.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/cat_images.zip or
        /content/cat_images.zip.zip, and cannot find /content/cat_images.zip.ZIP, period.


In [ ]:
!pip install -q ultralytics transformers

In [ ]:
# 1. 필수 라이브러리 설치 및 깃허브 자동 복구
!pip install -q ultralytics transformers

import os
# 만약 SafeSight 폴더가 없으면 자동으로 깃허브에서 다시 복사해옵니다.
if not os.path.exists('/content/SafeSight'):
    print("🔄 사라진 깃허브 저장소를 다시 가져오는 중...")
    # 지영님의 실제 깃허브 주소로 연동해두시면 됩니다!
    !git clone -b feat/YOLO https://github.com/Yeondu428/SafeSight.git
else:
    print("✅ 깃허브 저장소가 이미 존재합니다.")

# 2. 고양이 압축 해제
if not os.path.exists('/content/mydataset/cat'):
    print("📦 고양이 이미지 압축 푸는 중...")
    !unzip -q /content/cat_images.zip -d /content/mydataset/cat
else:
    print("✅ 고양이 이미지 압축 해제가 이미 완료되었습니다.")

# 3. 모델 및 전처리 인프라 가동
import glob
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO
from transformers import CLIPProcessor, CLIPModel

# GPU 장치 자동 세팅 (T4 가속기 작동 확인)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 현재 최종 연산 장치: {device} (cuda로 떠야 정상 속도가 나옵니다!)")

# 가중치 파일 로드
model_path = '/content/SafeSight/models/best.pt'
yolo_model = YOLO(model_path).to(device)

# CLIP 모델 로드
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# 4. 진짜 고양이 사진 폴더 긁어오기 (0장 오류 원천 차단!)
image_dir = '/content/mydataset/cat'
image_paths = glob.glob(os.path.join(image_dir, "*"))
print(f"📸 발견된 고양이 이미지 개수: {len(image_paths)}장! 임베딩 데이터베이스 구축을 시작합니다.")

all_embeddings = []
valid_image_paths = []

# tqdm 게이지 바를 보며 안정적으로 크롭 + 전처리 + 벡터 추출 진행
for path in tqdm(image_paths, desc="Processing Cat Datasets"):
    try:
        img = Image.open(path).convert('RGB')

        # [전처리 A] YOLO를 통한 동물 구역 정밀 크롭
        yolo_results = yolo_model(img, verbose=False)
        boxes = yolo_results[0].boxes
        cropped_img = None

        for box in boxes:
            conf = float(box.conf[0])
            if conf >= 0.4:  # 확신도 40% 이상 구역만 싹둑
                xyxy = box.xyxy[0].tolist()
                cropped_img = img.crop((int(xyxy[0]), int(xyxy[1]), int(xyxy[2]), int(xyxy[3])))
                break

        if cropped_img is None:
            cropped_img = img  # 미감지 시 원본 전체 구역 유지

        # [전처리 B & C] CLIP 표준 해상도(224x224) 통일 및 코사인 유사도 정규화
        cropped_img = cropped_img.resize((224, 224))
        inputs = processor(images=cropped_img, return_tensors="pt").to(device)

        with torch.no_grad():
            image_features = clip_model.get_image_features(**inputs)
            # 코사인 유사도 전용 L2 정규화 전처리
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            feature_numpy = image_features.cpu().numpy()[0]

        all_embeddings.append(feature_numpy)
        valid_image_paths.append(path)
    except:
        continue

# 5. 넘파이 벡터 파일 최종 저장
np.save('/content/cat_embeddings.npy', np.array(all_embeddings, dtype=np.float32))
np.save('/content/cat_paths.npy', np.array(valid_image_paths))

print("\n🎉 고양이 벡터 데이터베이스(DB) 구축이 완벽하게 끝났습니다!")
print(f"💾 결과 저장 성공! 행렬 크기: {np.array(all_embeddings).shape}")

🔄 사라진 깃허브 저장소를 다시 가져오는 중...
Cloning into 'SafeSight'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 68 (delta 19), reused 21 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 6.44 MiB | 11.82 MiB/s, done.
Resolving deltas: 100% (19/19), done.
📦 고양이 이미지 압축 푸는 중...
[/content/cat_images.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/cat_images.zip or
        /content/cat_images.zip.zip, and cannot find /content/cat_images.zip.ZIP, period.
🚀 현재 최종 연산 장치: cuda (cuda로 떠야 정상 속도가 나옵니다!)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

📸 발견된 고양이 이미지 개수: 0장! 임베딩 데이터베이스 구축을 시작합니다.


Processing Cat Datasets: 0it [00:00, ?it/s]


🎉 고양이 벡터 데이터베이스(DB) 구축이 완벽하게 끝났습니다!
💾 결과 저장 성공! 행렬 크기: (0,)


In [ ]:
import os

# 1. 기존에 SafeSight 폴더가 있다면 삭제하고 새로 클론받아 최신 상태를 유지합니다.
!rm -rf /content/SafeSight
print("🔄 깃허브에서 최신 코드를 반영하여 저장소를 복사해오는 중...")

# 2. 지영님의 레포지토리와 브랜치를 지정해 클론합니다.
!git clone -b feat/YOLO https://github.com/Yeondu428/SafeSight.git

🔄 깃허브에서 최신 코드를 반영하여 저장소를 복사해오는 중...
Cloning into 'SafeSight'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 76 (delta 23), reused 21 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (76/76), 6.44 MiB | 11.55 MiB/s, done.
Resolving deltas: 100% (23/23), done.


In [ ]:
import os

print("📂 [최상위 루트 경로 파일 목록]")
root_files = os.listdir('/content/SafeSight')
print(root_files)
# 👉 이 목록에 방금 만드신 'app.py'(또는 streamlit_ui.py)가 들어있는지 확인하세요!

print("\n📦 [models/ 폴더 내부 파일 목록]")
if os.path.exists('/content/SafeSight/models'):
    model_files = os.listdir('/content/SafeSight/models')
    print(model_files)
    # 👉 이 목록에 best.pt, cat_embeddings.npy, cat_paths.npy가 다 있는지 확인하세요!
else:
    print("❌ models 폴더를 찾을 수 없습니다. 경로를 확인해주세요.")

📂 [최상위 루트 경로 파일 목록]
['notebooks', 'data', 'README.md', 'app', '.gitignore', '.git', 'streamlit_ui.py', 'models']

📦 [models/ 폴더 내부 파일 목록]
['results.png', '.gitkeep', 'cat_embeddings.npy', 'best.pt', 'cat_paths.npy']


In [ ]:
import numpy as np
from ultralytics import YOLO

print("⚙️ 깃허브에 올린 파일들을 가상으로 로드해봅니다...")

try:
    # 1. 가중치 파일 로드 테스트
    yolo_test = YOLO('/content/SafeSight/models/best.pt')
    print("──> 1. YOLO 가중치 파일(best.pt) 로드 성공!")

    # 2. 고양이 넘파이 파일 로드 테스트
    cat_embeds_test = np.load('/content/SafeSight/models/cat_embeddings.npy')
    cat_paths_test = np.load('/content/SafeSight/models/cat_paths.npy')
    print(f"──> 2. 고양이 임베딩 DB 로드 성공! (행렬 크기: {cat_embeds_test.shape})")

    # 3. 강아지 넘파이 파일 로드 테스트
    # ⚠️ 만약 강아지 파일명이 shelter_...가 아니라 다르면 이름만 바꿔서 테스트하세요!
    dog_embeds_test = np.load('/content/SafeSight/models/shelter_embeddings.npy')
    dog_paths_test = np.load('/content/SafeSight/models/shelter_paths.npy')
    print(f"──> 3. 강아지 임베딩 DB 로드 성공! (행렬 크기: {dog_embeds_test.shape})")

    print("\n✨ 🎉 ✅ 깃허브 파일들 검증 완료! 스트림릿 클라우드로 배포하셔도 좋습니다!")

except Exception as e:
    print(f"\n❌ 에러 발생: {e}")
    print("💡 에러 메시지를 보고 깃허브에 파일이 누락되었거나 이름이 다른지 확인해주세요.")

⚙️ 깃허브에 올린 파일들을 가상으로 로드해봅니다...
──> 1. YOLO 가중치 파일(best.pt) 로드 성공!
──> 2. 고양이 임베딩 DB 로드 성공! (행렬 크기: (0,))

❌ 에러 발생: [Errno 2] No such file or directory: '/content/SafeSight/models/shelter_embeddings.npy'
💡 에러 메시지를 보고 깃허브에 파일이 누락되었거나 이름이 다른지 확인해주세요.


In [ ]:
!rm -rf /content/cat_images.zip
!rm -rf /content/mydataset/cat

In [2]:
# 1. 필수 라이브러리 재설치 (세션 초기화 대응)
print("⚙️ 1단계: 필수 패키지(YOLO, CLIP)를 설치합니다...")
!pip install -q ultralytics transformers

import os
import glob
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO
from transformers import CLIPProcessor, CLIPModel

# 2. 깃허브 저장소 확인 및 다운로드
if not os.path.exists('/content/SafeSight'):
    print("\n🔄 2단계: 깃허브에서 SafeSight 저장소를 가져오는 중...")
    !git clone -b feat/YOLO https://github.com/Yeondu428/SafeSight.git
else:
    print("\n✅ 2단계: SafeSight 저장소가 이미 준비되어 있습니다.")

# 3. 고양이 이미지 압축 해제 및 경로 검증
print("\n📦 3단계: 고양이 이미지 압축 푸는 중... (잠시만 기다려주세요)")
!rm -rf /content/mydataset/cat
!mkdir -p /content/mydataset/cat
!unzip -q /content/cat_images.zip -d /content/mydataset/cat

# 모든 하위 폴더 내부의 이미지 탐색
search_pattern = '/content/mydataset/cat/**/*'
all_files = glob.glob(search_pattern, recursive=True)
image_paths = [f for f in all_files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print("\n📊 [경로 검증 결과]")
print(f"──> 실제로 압축이 풀려 나온 총 파일 개수: {len(all_files)}개")
print(f"──> 🎯 AI 모델이 처리할 고양이 이미지 개수: {len(image_paths)}장")

# 4. 이미지 탐색 성공 시 AI 연산 가동
if len(image_paths) == 0:
    print("\n❌ [경고] 이미지 개수가 0장입니다. 파일 창(📁)에 cat_images.zip이 완벽히 업로드되었는지, 이름이 맞는지 확인해주세요.")
else:
    print(f"\n✅ 확인 완료! {len(image_paths)}장으로 임베딩 벡터 데이터베이스 구축을 시작합니다. 🚀")
    print("-" * 50)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🖥️ 현재 가속 연산 장치: {device}")

    # 가중치 파일 로드
    model_path = '/content/SafeSight/models/best.pt'
    yolo_model = YOLO(model_path).to(device)

    # CLIP 멀티모달 모델 로드
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    all_embeddings = []
    valid_image_paths = []

    # 크롭 + 임베딩 전처리 루프 가동
    for path in tqdm(image_paths, desc="Processing Cat Datasets"):
        try:
            img = Image.open(path).convert('RGB')

            # YOLO 실시간 동물 구역 크롭
            yolo_results = yolo_model(img, verbose=False)
            boxes = yolo_results[0].boxes
            cropped_img = None

            for box in boxes:
                if float(box.conf[0]) >= 0.4:
                    xyxy = box.xyxy[0].tolist()
                    cropped_img = img.crop((int(xyxy[0]), int(xyxy[1]), int(xyxy[2]), int(xyxy[3])))
                    break

            if cropped_img is None:
                cropped_img = img

            # CLIP 규격화 및 코사인 정규화
            cropped_img = cropped_img.resize((224, 224))
            inputs = processor(images=cropped_img, return_tensors="pt").to(device)

            with torch.no_grad():
                image_features = clip_model.get_image_features(**inputs)
                image_features = image_features / image_features.norm(dim=-1, keepdim=True)
                feature_numpy = image_features.cpu().numpy()[0]

            all_embeddings.append(feature_numpy)
            valid_image_paths.append(path)
        except:
            continue

    # 5. 넘파이 데이터베이스 파일 최종 빌드 및 저장
    np.save('/content/cat_embeddings.npy', np.array(all_embeddings, dtype=np.float32))
    np.save('/content/cat_paths.npy', np.array(valid_image_paths))

    print("\n🎉 고양이 벡터 데이터베이스(DB) 구축 완료!")
    print(f"💾 결과 저장 성공! 행렬 크기: {np.array(all_embeddings).shape}")

⚙️ 1단계: 필수 패키지(YOLO, CLIP)를 설치합니다...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

🔄 2단계: 깃허브에서 SafeSight 저장소를 가져오는 중...
Cloning into 'SafeSight'...
remote: Enumerating objects: 76, done.
remote: Counting objects: 100% (76/76), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 76 (delta 23), reused 21 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (76/76), 6.44 MiB | 7.72 MiB/s, done.
Resolving deltas: 100% (23/23), done.

📦 3단계: 고양이 이미지 압축 푸는 중... (잠시만 기다려주세요)

📊 [경로 검증 결과]
──> 실제로 압축이 풀려 나온 총 파일 개수: 14787개
──> 🎯 AI 모델이 처리할 고양이 이미지 개수: 1477

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Processing Cat Datasets: 100%|██████████| 14779/14779 [10:09<00:00, 24.23it/s]



🎉 고양이 벡터 데이터베이스(DB) 구축 완료!
💾 결과 저장 성공! 행렬 크기: (0,)
